# MP-PSRP Notebook Explorer

Расширенный notebook для исследования среды, масок, rollout-трассировки и короткого цикла обучения.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from omegaconf import OmegaConf

from src.train.notebook import (
    build_components,
    episode_summary,
    episode_to_frame,
    history_to_frame,
    load_config,
    plot_action_masks,
    plot_episode_dashboard,
    plot_instance_map,
    plot_inventory_heatmap,
    plot_training_history,
    rollout_episode,
    station_summary_frame,
    vehicle_summary_frame,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.precision", 3)

## 1. Build Components

Для быстрых итераций используется `small`-конфиг и короткий train loop. Эти overrides можно менять прямо из notebook.

In [ ]:
cfg = load_config([
    "env=small",
    "algo=ppo",
    "experiment.total_iterations=6",
    "experiment.rollout_length=12",
    "experiment.eval_interval=2",
])

env, policy, algo, trainer = build_components(cfg)
obs, info = env.reset()

print(OmegaConf.to_yaml(cfg))

## 2. Initial State Snapshot

Сначала смотрим на интерпретируемые таблицы по станциям и машинам, а потом на геометрию инстанса и action masks.

In [ ]:
display(pd.Series(info, name="reset_info"))
display(station_summary_frame(env, obs))
display(vehicle_summary_frame(env, obs))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
plot_instance_map(env, ax=axes[0], annotate=True)
image = plot_inventory_heatmap(
    env,
    env.state.station_inventory,
    ax=axes[1],
    title="Initial Inventory Fill Ratio",
)
fig.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04)
plt.show()

In [ ]:
plot_action_masks(obs)
plt.show()

## 3. Baseline Rollout Before Training

Запускаем один детерминированный rollout текущей политики и смотрим маршрут, reward-компоненты и кумулятивные метрики.

In [ ]:
baseline_episode = rollout_episode(env, policy, deterministic=True, max_steps=20)
baseline_frame = episode_to_frame(baseline_episode)

display(pd.DataFrame([episode_summary(baseline_episode)], index=["before_training"]))
display(baseline_frame.head(10))

In [ ]:
plot_episode_dashboard(env, baseline_episode)
plt.show()

## 4. Short Training Run

Это не полноценный эксперимент, а smoke-run прямо из notebook: собираем немного rollout-ов, обновляем PPO и смотрим динамику лоссов и eval-метрик.

In [ ]:
history = trainer.train()
history_frame = history_to_frame(history)

display(history_frame)
plot_training_history(history)
plt.show()

## 5. Rollout After Training

Повторяем тот же exploratory rollout и сравниваем агрегаты до и после краткого обучения.

In [ ]:
trained_episode = rollout_episode(env, policy, deterministic=True, max_steps=20)
trained_frame = episode_to_frame(trained_episode)

comparison = pd.DataFrame(
    [episode_summary(baseline_episode), episode_summary(trained_episode)],
    index=["before_training", "after_training"],
)

display(comparison)
display(trained_frame.head(10))

In [ ]:
plot_episode_dashboard(env, trained_episode)
plt.show()

## 6. Useful Next Tweaks

- Переключить алгоритм: `algo=d3po` или `algo=ppo_lagrangian` в `load_config([...])`
- Увеличить горизонт и размерность: `env=medium` или `env=real_case`
- Для D3PO можно передать preference vector в `rollout_episode(..., preferences=np.array([...], dtype=np.float32))`
- Для paper-aligned reward shaping включить `env.enable_potential_shaping=true`